# Eniac A/B Test: Finding the Best Call-to-Action Button

**Case:** Eniac homepage CTA button experiment  
**Methods:** Chi-Square test of independence, Bonferroni-corrected pairwise post-hoc tests, relative-lift analysis.

This notebook documents the statistical analysis of an A/B/C/D test run on the Eniac landing page. Four variants of the primary call-to-action button were served to randomly assigned visitors. The goal is to identify the variant that maximizes click-through rate (CTR) while respecting statistical rigor.

## 1. Context

Eniac is an online electronics retailer. The marketing team hypothesized that the current CTA button on the homepage is underperforming and designed four candidate variants crossing two design dimensions:

| Version | Color | Label       |
|:-------:|:-----:|:------------|
| A       | White | SHOP NOW    |
| B       | Red   | SHOP NOW    |
| C       | White | SEE DEALS   |
| D       | Red   | SEE DEALS   |

Each visitor is randomly assigned to exactly one variant. We record the number of unique visitors and the number of CTA clicks per variant.

## 2. Data Preparation

We load the four CSV files — one per variant — and extract the number of clicks on the primary CTA button of each page.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from itertools import combinations

pd.set_option("display.max_colwidth", None)

In [ ]:
DATA_DIR = "../data"

eniac_a = pd.read_csv(f"{DATA_DIR}/eniac_a.csv")
eniac_b = pd.read_csv(f"{DATA_DIR}/eniac_b.csv")
eniac_c = pd.read_csv(f"{DATA_DIR}/eniac_c.csv")
eniac_d = pd.read_csv(f"{DATA_DIR}/eniac_d.csv")

eniac_a.head()

### 2.1 Extract clicks and visits

For variants A and B the primary CTA is labelled `SHOP NOW`; for variants C and D it is labelled `SEE DEALS`. Visitor counts were read from the snapshot row (element ID 25) of each export.

In [ ]:
clicks_a = eniac_a.loc[eniac_a["Name"] == "SHOP NOW", "No. clicks"].values[0]
clicks_b = eniac_b.loc[eniac_b["Name"] == "SHOP NOW", "No. clicks"].values[0]
clicks_c = eniac_c.loc[eniac_c["Name"] == "SEE DEALS", "No. clicks"].values[0]
clicks_d = eniac_d.loc[eniac_d["Name"] == "SEE DEALS", "No. clicks"].values[0]

visits = {"A": 25326, "B": 24747, "C": 24876, "D": 25233}

contingency_table = [
    [clicks_a, visits["A"] - clicks_a],
    [clicks_b, visits["B"] - clicks_b],
    [clicks_c, visits["C"] - clicks_c],
    [clicks_d, visits["D"] - clicks_d],
]
contingency_table

### 2.2 Click-Through Rates

The CTR is the share of visitors who clicked the CTA.

In [ ]:
ctr_a = clicks_a / visits["A"]
ctr_b = clicks_b / visits["B"]
ctr_c = clicks_c / visits["C"]
ctr_d = clicks_d / visits["D"]

ctr_summary = pd.DataFrame(
    {
        "Version": ["A", "B", "C", "D"],
        "Color": ["White", "Red", "White", "Red"],
        "Label": ["SHOP NOW", "SHOP NOW", "SEE DEALS", "SEE DEALS"],
        "Visits": list(visits.values()),
        "Clicks": [clicks_a, clicks_b, clicks_c, clicks_d],
        "CTR (%)": [round(ctr * 100, 2) for ctr in (ctr_a, ctr_b, ctr_c, ctr_d)],
    }
)
ctr_summary

## 3. Hypothesis Testing

### 3.1 Hypotheses

- **H0 (Null):** All four versions share the same true CTR.
- **H1 (Alternative):** At least one version has a different true CTR.

### 3.2 Significance level

A relatively permissive significance level of `alpha = 0.10` was agreed with the business. The business accepts a 10% probability of a type-I error in exchange for higher statistical power.

In [ ]:
alpha = 0.10

### 3.3 Global Chi-Square test

We run a Pearson chi-square test of independence on the 4×2 contingency table (clicks vs. non-clicks by version).

In [ ]:
chi2, p_value, dof, expected = stats.chi2_contingency(contingency_table)

print(f"Chi-Square statistic: {chi2:.4f}")
print(f"P-value:              {p_value:.4e}")
print(f"Degrees of freedom:   {dof}")
print("Expected frequencies (rounded):")
print(expected.round(2))

**Interpretation.** The p-value is far below our alpha, so we reject H0. The four variants do **not** share the same CTR — at least one button performs differently from the others. The global test alone, however, does not tell us *which* variant wins. That requires a post-hoc analysis.

## 4. Post-hoc Analysis (Pairwise Chi-Square with Bonferroni Correction)

With four variants we have `C(4, 2) = 6` pairwise comparisons. Running six independent tests at `alpha = 0.10` inflates the family-wise error rate well above 10%. We apply the Bonferroni correction:

$$\alpha_{adj} = \frac{\alpha}{k} = \frac{0.10}{6} \approx 0.0167$$

In [ ]:
versions = ["A", "B", "C", "D"]
num_comparisons = 6
alpha_bonf = alpha / num_comparisons

print(f"Bonferroni-adjusted alpha: {alpha_bonf:.4f}\n")

for i, j in combinations(range(4), 2):
    pair_table = [contingency_table[i], contingency_table[j]]
    _, p_pair, _, _ = stats.chi2_contingency(pair_table)
    verdict = "SIGNIFICANT" if p_pair < alpha_bonf else "not significant"
    print(f"{versions[i]} vs {versions[j]}: p = {p_pair:.4e} -> {verdict}")

**Findings.**

- Both white buttons (A, C) significantly outperform both red buttons (B, D).
- The difference between the two white buttons (A vs. C) is **not** statistically significant — we cannot claim that `SEE DEALS` beats `SHOP NOW` on CTR alone.
- **Color is the dominant driver** of CTR; the label is a secondary effect.

## 5. Business Perspective: Relative Lift

Statistical significance answers "is the difference real?" but not "is it worth it?". The relative lift translates the CTR gap into a business-friendly percentage improvement over the current baseline (A).

In [ ]:
lift_c_vs_a = (ctr_c - ctr_a) / ctr_a * 100
print(f"Relative lift of Version C over Version A: {lift_c_vs_a:.2f}%")

Version C delivers a ~5% relative lift in CTR over the current baseline A. Although this gap is not statistically significant on its own, it is directionally positive and economically meaningful at Eniac's traffic scale.

## 6. Supplementary Metrics: Drop-off and Return Rate

Two further metrics were reported in the LMS material (no CSV export available) and reinforce the quantitative picture:

- **Drop-off rate** — share of users who leave the landing page without continuing. *Lower is better.*
- **Homepage-return rate** — share of users who return to the homepage after clicking the CTA; a proxy for misleading or disappointing clicks. *Lower is better.*

Version C has the lowest drop-off and the lowest return rate among the variants with reliable tracking. Version B data is missing due to a tracking error during the test window.

In [ ]:
supplementary = {
    "Version": ["A", "B", "C", "D"],
    "CTR": [2.02, 1.14, 2.12, 0.76],
    "Drop_off": [38.5, np.nan, 18.2, 61.8],
    "Return_rate": [4.1, np.nan, 2.3, 4.6],
}
metrics_df = pd.DataFrame(supplementary)

plt.style.use("seaborn-v0_8-muted")
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

winner_color = "#2ecc71"
others_color = "#bdc3c7"

def bar_palette(versions_series):
    return [winner_color if v == "C" else others_color for v in versions_series]

axes[0].bar(metrics_df["Version"], metrics_df["CTR"], color=bar_palette(metrics_df["Version"]))
axes[0].set_title("Click-Through Rate\n(Higher = Better)", fontweight="bold")
axes[0].set_ylabel("Percentage (%)")
for i, value in enumerate(metrics_df["CTR"]):
    axes[0].text(i, value + 0.05, f"{value}%", ha="center", fontweight="bold")

drop_df = metrics_df.dropna(subset=["Drop_off"]).reset_index(drop=True)
axes[1].bar(drop_df["Version"], drop_df["Drop_off"], color=bar_palette(drop_df["Version"]))
axes[1].set_title("Drop-off Rate\n(Lower = Better)", fontweight="bold")
axes[1].set_ylabel("Percentage (%)")
for i, value in enumerate(drop_df["Drop_off"]):
    axes[1].text(i, value + 1, f"{value}%", ha="center", fontweight="bold")

ret_df = metrics_df.dropna(subset=["Return_rate"]).reset_index(drop=True)
axes[2].bar(ret_df["Version"], ret_df["Return_rate"], color=bar_palette(ret_df["Version"]))
axes[2].set_title("Homepage-Return Rate\n(Lower = Better)", fontweight="bold")
axes[2].set_ylabel("Percentage (%)")
for i, value in enumerate(ret_df["Return_rate"]):
    axes[2].text(i, value + 0.1, f"{value}%", ha="center", fontweight="bold")

plt.suptitle(
    "Eniac A/B Test Results — Version C is the recommended variant",
    fontsize=16,
    fontweight="bold",
    y=1.03,
)
plt.tight_layout()
plt.show()

## 7. Business Recommendation

**Recommendation: Roll out Version C — the white `SEE DEALS` button.**

1. **Statistical evidence.** The global chi-square test rejects the null at `alpha = 0.10`. Pairwise Bonferroni-corrected tests show that both white buttons (A, C) significantly beat both red buttons (B, D).
2. **Direction of effect.** Among the white buttons, C has the highest observed CTR (2.12%) and a ~5% relative lift over the current baseline A. The A-vs-C gap is not statistically significant, so this is a judgment call rather than a forced choice.
3. **Supplementary metrics.** C also shows the lowest drop-off rate and the lowest homepage-return rate, indicating that the `SEE DEALS` label sets user expectations more accurately than `SHOP NOW`.
4. **Risk.** Rolling out C instead of A carries minimal risk: both are statistically equivalent on CTR, and C dominates on every supporting metric we can measure.

**Next steps.** Monitor post-launch CTR and downstream conversion for two weeks; if the lift holds, keep C permanently. If Eniac wants to resolve the A-vs-C question definitively, a follow-up A/B test with a larger sample on just those two variants would provide the statistical power needed.